In [3]:
from  datasets import load_dataset
ds = load_dataset("stanford-oval/churro-dataset", split="train")

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/223 [00:00<?, ?it/s]

In [5]:
import os 

base_url = "https://api.portkey.ai/v1"
api_key = os.getenv('PORTKEY_API_KEY')
model = "gpt-5"  # or "qwen3-vl-flash" for faster/cheaper

In [6]:
api_key

In [4]:
# @title OCR Processing Functions
import asyncio
import aiohttp
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

client = AsyncOpenAI(api_key=api_key, base_url=base_url)

async def transcribe_image(img: tuple, session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Process a single image with rate limiting."""
    async with semaphore:
        image_path, image_b64 = img

        try:
            completion = await asyncio.wait_for(
                client.chat.completions.create(
                    model=model,
                    messages=[{
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "Extract all text."
                            },
                            {
                                "type": "image_url",
                                "image_url": {"url": image_b64}
                            }
                        ]
                    }],
                ),
                timeout=360
            )

            return {
                "name": image_path.name,
                "text": completion.choices[0].message.content
            }
        except asyncio.TimeoutError:
            print(f"⏱️  Timeout: {image_path.name}")
            return None
        except Exception as e:
            print(f"❌ Error on {image_path.name}: {e}")
            return None

async def transcribe_all_images(images, max_concurrent=15):
    """Process all images with concurrency control."""
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    timeout = aiohttp.ClientTimeout(total=300)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        semaphore = asyncio.Semaphore(max_concurrent)

        # Encode images with progress
        print("📊 Encoding images...")
        images_list = list(tqdm(images, desc="Encoding"))

        # Process with progress
        tasks = [transcribe_image(img, session, semaphore) for img in images_list]
        results = []

        for task in tqdm.as_completed(tasks, total=len(tasks), desc="Transcribing"):
            result = await task
            if result:
                results.append(result)

    return results

NameError: name 'api_key' is not defined